# Merge and extraction of Mission Gate videos

In [ ]:
#!/usr/bin/env python3
"""
Merge multiple CSV files and summarize frame ranges by seq and cam_view.

Requirements:
- Python 3
- pandas (`pip install pandas`)
"""

import pandas as pd
import glob
import os

# -------------------- USER SETTINGS --------------------
INPUT_CSV_FOLDER = "../data/csv_logs"  # Folder containing multiple CSV files
OUTPUT_CSV_FILE = "../data/csv_logs/merged_summary.csv" # Output CSV file
FILTER_SEQ = None       # e.g., "seq1" or None for all
FILTER_CAM_VIEW = None  # e.g., "cam1" or None for all
# -------------------------------------------------------

# Step 1: Read all CSV files
all_files = glob.glob(os.path.join(INPUT_CSV_FOLDER, "*.csv"))
if not all_files:
    print("No CSV files found in folder.")
    exit(1)

df_list = [pd.read_csv(f) for f in all_files]
df = pd.concat(df_list, ignore_index=True)

# Step 2: Filter seq and cam_view if specified
if FILTER_SEQ:
    df = df[df["seq"] == FILTER_SEQ]
if FILTER_CAM_VIEW:
    df = df[df["cam_view"] == FILTER_CAM_VIEW]

# Step 3: Group by seq and cam_view
grouped = df.groupby(["seq", "cam_view"])

summary_rows = []

for (seq, cam_view), group in grouped:
    # Determine min and max frame_num
    start_frame = group["frame_num"].min()
    end_frame = group["frame_num"].max()
    
    # Get first URL (assuming all rows in group are same video)
    url = group["url"].iloc[0]
    
    # Transfer additional info (assuming consistent within group)
    gait_event = group["gait_event"].iloc[0] if "gait_event" in group else ""
    dataset = group["dataset"].iloc[0] if "dataset" in group else ""
    gait_pat = group["gait_pat"].iloc[0] if "gait_pat" in group else ""
    
    summary_rows.append({
        "seq": seq,
        "cam_view": cam_view,
        "start_frame": start_frame,
        "end_frame": end_frame,
        "url": url,
        "gait_event": gait_event,
        "dataset": dataset,
        "gait_pat": gait_pat
    })

# Step 4: Save to new CSV
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_CSV_FILE, index=False)
print(f"Summary CSV saved to {OUTPUT_CSV_FILE}")


# Integrated Downloader version

In [ ]:
#!/usr/bin/env python3
"""
Parallel, resumable YouTube frame-based segment downloader
with checksum logging.

Requirements:
- Python 3.9+
- pandas
- yt-dlp
- ffmpeg
- Optional but recommended: deno
"""

import os
import subprocess
import hashlib
import pandas as pd
from yt_dlp import YoutubeDL
from concurrent.futures import ProcessPoolExecutor, as_completed

# -------------------- USER SETTINGS --------------------
INPUT_CSV = "../data/csv_logs/merged_summary_test.csv"
OUTPUT_CSV = "../data/csv_logs/MissionGait/merged_summary_enriched.csv"

TEMP_FOLDER = "../data/temp_videos"
OUTPUT_VIDEO_FOLDER = "../data/video_snippets"

MAX_WORKERS = 4        # adjust for your machine
# ------------------------------------------------------

os.makedirs(TEMP_FOLDER, exist_ok=True)
os.makedirs(OUTPUT_VIDEO_FOLDER, exist_ok=True)

# ------------------------------------------------------
# Utilities
# ------------------------------------------------------

def frame_to_timestamp(frame, fps):
    seconds = frame / fps
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = seconds % 60
    return f"{h:02}:{m:02}:{s:06.3f}"

def sha256_checksum(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

def get_video_info(url):
    with YoutubeDL({
        "quiet": True,
        "skip_download": True,
        "remote_components": "ejs:github",
    }) as ydl:
        return ydl.extract_info(url, download=False)

def download_full_video(url, title):
    with YoutubeDL({
        "format": "bestvideo[ext=mp4]+bestaudio[ext=m4a]/best",
        "merge_output_format": "mp4",
        "outtmpl": os.path.join(TEMP_FOLDER, f"{title}.%(ext)s"),
        "remote_components": "ejs:github",
        "noplaylist": True,
        "quiet": True,
    }) as ydl:
        ydl.download([url])

def cut_segment(input_file, start_ts, end_ts, output_file):
    subprocess.run([
        "ffmpeg", "-y",
        "-i", input_file,
        "-ss", start_ts,
        "-to", end_ts,
        "-c:v", "libx264",
        "-c:a", "aac",
        output_file
    ], check=True)

# ------------------------------------------------------
# Worker
# ------------------------------------------------------

def process_row(idx, row):
    try:
        output_name = (
            f"{row.seq}_{row.cam_view}_"
            f"{row.gait_event}_{row.dataset}_{row.gait_pat}.mp4"
        )
        output_path = os.path.join(OUTPUT_VIDEO_FOLDER, output_name)

        # Resume: skip if already processed
        if os.path.exists(output_path) and not pd.isna(row.get("checksum", None)):
            return idx, None

        info = get_video_info(row.url)
        title = info["title"]
        uploader = info.get("uploader", "")

        fps_list = [f.get("fps") for f in info.get("formats", []) if f.get("fps")]
        fps = max(fps_list) if fps_list else 30

        start_ts = frame_to_timestamp(row.start_frame, fps)
        end_ts = frame_to_timestamp(row.end_frame, fps)
        duration = round((row.end_frame - row.start_frame) / fps, 3)

        # Download full video
        download_full_video(row.url, title)
        input_video = os.path.join(TEMP_FOLDER, f"{title}.mp4")

        # Cut snippet
        cut_segment(input_video, start_ts, end_ts, output_path)

        # Cleanup temp
        if os.path.exists(input_video):
            os.remove(input_video)

        checksum = sha256_checksum(output_path)

        return idx, {
            "title": title,
            "uploader": uploader,
            "fps": fps,
            "start_time": start_ts,
            "end_time": end_ts,
            "duration": duration,
            "checksum": checksum
        }

    except Exception as e:
        return idx, {"error": str(e)}

# ------------------------------------------------------
# Main
# ------------------------------------------------------

df = pd.read_csv(INPUT_CSV)

# Ensure columns exist (resume-safe)
for col in [
    "title", "uploader", "fps",
    "start_time", "end_time", "duration", "checksum"
]:
    if col not in df.columns:
        df[col] = ""

tasks = []

#Blocked out for testing because ProcessPoolExecutor needs to run at Top level in py script and cant be inside a Jupyter notebook cell
"""
with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
    for idx, row in df.iterrows():
        tasks.append(executor.submit(process_row, idx, row))

    for future in as_completed(tasks):
        idx, result = future.result()

        if result is None:
            continue

        if "error" in result:
            print(f"Row {idx} failed: {result['error']}")
            continue

        for k, v in result.items():
            df.at[idx, k] = v
"""
# Inserted for testing as alternative to the above
for idx, row in df.iterrows():
    idx, result = process_row(idx, row)
    
    if result is None:
        continue
    if "error" in result:
        print(f"Row {idx} failed: {result['error']}")
        continue
    
    for k, v in result.items():
        df.at[idx, k] = v


# Save enriched CSV
df.to_csv(OUTPUT_CSV, index=False)
print(f"\n✅ Finished. CSV written to {OUTPUT_CSV}")


### Removes an extra heading if necessary and formats the csv

In [ ]:
import pandas as pd
INPUT_CSV = "../data/csv_logs/merged_summary_test.csv"
df_raw = pd.read_csv(INPUT_CSV)

df = df_raw["merged_summary"].str.split(";", expand=True)

df.columns = [
    "seq",
    "cam_view",
    "start_frame",
    "end_frame",
    "url",
    "gait_event",
    "dataset",
    "gait_pat",
]

# 🔧 Remove rows where start_frame is not numeric (e.g. header rows)
df = df[df["start_frame"].str.isnumeric()]

# Convert numeric columns
df["start_frame"] = df["start_frame"].astype(int)
df["end_frame"] = df["end_frame"].astype(int)

# Validate required columns
required = {"seq", "cam_view", "start_frame", "end_frame", "url"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {missing}")

# Overwrite original file
df.to_csv(INPUT_CSV, index=False)


# Reformats the csv if necessary

In [ ]:
#adjust the csv
import pandas as pd

INPUT_CSV = "../data/csv_logs/merged_summary_test.csv"

# Load CSV
df_raw = pd.read_csv(INPUT_CSV)

# Detect merged column
if len(df_raw.columns) == 1:
    merged_col = df_raw.columns[0]
    print(f"Detected single merged column: '{merged_col}'")

    # Split by semicolon (adjust if your CSV uses commas)
    df = df_raw[merged_col].str.split(";", expand=True)

    df.columns = [
        "seq",
        "cam_view",
        "start_frame",
        "end_frame",
        "url",
        "gait_event",
        "dataset",
        "gait_pat",
    ]
else:
    df = df_raw.copy()
    print("CSV already has multiple columns, no split needed.")

# 🔧 Remove rows where start_frame is not numeric
df = df[df["start_frame"].apply(lambda x: str(x).isnumeric())]

# Convert numeric columns
df["start_frame"] = df["start_frame"].astype(int)
df["end_frame"] = df["end_frame"].astype(int)

# Overwrite original CSV
df.to_csv(INPUT_CSV, index=False)
print(f"✅ Cleaned CSV saved to {INPUT_CSV}")
print(df.head())


## Runs all videos in the list

In [ ]:
#!/usr/bin/env python3
"""
YouTube Segment Downloader with incremental CSV enrichment

- Downloads video if missing
- Cuts QuickTime-compatible MP4 clips
- Extracts metadata from downloaded or existing videos
- Updates enriched CSV row by row, even if video exists
"""

from yt_dlp import YoutubeDL
import os
import subprocess
import pandas as pd
import re
import hashlib
import json
import csv

# -------------------- USER SETTINGS --------------------
INPUT_CSV = "../data/csv_logs/merged_summary_test.csv"
OUTPUT_CSV = "../data/csv_logs/merged_summary_enriched.csv"
TEMP_FOLDER = "../data/temp_videos"
OUTPUT_VIDEO_FOLDER = "../data/video_snippets"
# ------------------------------------------------------

os.makedirs(TEMP_FOLDER, exist_ok=True)
os.makedirs(OUTPUT_VIDEO_FOLDER, exist_ok=True)

# -------------------- Utilities ----------------------

def safe_name(s):
    return re.sub(r"[^\w\-_. ]", "_", str(s)).strip()

def frame_to_timestamp(frame, fps):
    seconds = frame / fps
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = seconds % 60
    return f"{h:02}:{m:02}:{s:06.3f}"

def sha256_checksum(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

def ffprobe_metadata(file_path):
    cmd = [
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration:stream=codec_type,codec_name,width,height,r_frame_rate",
        "-print_format", "json",
        file_path
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"ffprobe failed on {file_path}")
    return json.loads(result.stdout)

def download_full_video(url):
    ydl_opts = {
        "format": "bv*/b",  # best video or best combined
        "outtmpl": os.path.join(TEMP_FOLDER, "%(title)s.%(ext)s"),
        "noplaylist": True,
        "quiet": False,
        "remote_components": ["ejs:github"],
    }
    with YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=True)

    if info is None:
        raise RuntimeError("yt-dlp returned no info")
    if info.get("vcodec") == "none":
        raise RuntimeError("Audio-only stream — no video available")

    title = safe_name(info["title"])
    ext = info.get("ext")
    input_path = os.path.join(TEMP_FOLDER, f"{title}.{ext}")

    if not os.path.exists(input_path):
        raise FileNotFoundError(f"Downloaded file missing: {input_path}")

    return info, input_path

def cut_and_reencode(input_file, start_ts, end_ts, output_file):
    if not os.path.exists(output_file):
        subprocess.run([
            "ffmpeg", "-y",
            "-i", input_file,
            "-ss", start_ts,
            "-to", end_ts,
            "-c:v", "libx264",
            "-c:a", "aac",
            output_file
        ], check=True)

def append_row_to_csv(row_dict, csv_file):
    file_exists = os.path.exists(csv_file)
    with open(csv_file, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=row_dict.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(row_dict)

# -------------------- Main ---------------------------

def main():
    df = pd.read_csv(INPUT_CSV)

    # Add enrichment columns if missing
    for col in ["title","uploader","fps","start_time","end_time","duration","checksum","width","height"]:
        if col not in df.columns:
            df[col] = ""

    for idx, row in df.iterrows():
        try:
            print(f"\n▶ Processing row {idx}")

            if pd.isna(row.url) or pd.isna(row.start_frame) or pd.isna(row.end_frame):
                print("Skipping row due to missing URL or frames")
                continue

            start_frame = int(row.start_frame)
            end_frame = int(row.end_frame)

            # Build output filename
            output_name = safe_name(f"{row.seq}_{row.cam_view}_{row.gait_event}_{row.dataset}_{row.gait_pat}.mp4")
            output_path = os.path.join(OUTPUT_VIDEO_FOLDER, output_name)

            # Download video only if missing
            input_video = None
            if not os.path.exists(output_path):
                info, input_video = download_full_video(row.url)
                # Cut clip
                fps_list = [f.get("fps") for f in info.get("formats", []) if f.get("fps")]
                fps = max(fps_list) if fps_list else 30
                start_ts = frame_to_timestamp(start_frame, fps)
                end_ts = frame_to_timestamp(end_frame, fps)
                cut_and_reencode(input_video, start_ts, end_ts, output_path)
                if os.path.exists(input_video):
                    os.remove(input_video)
            else:
                print("Video already exists, skipping download/cut")
                # Still need info for CSV
                try:
                    info, _ = download_full_video(row.url)
                except Exception as e:
                    info = {"title": row.seq, "uploader": "", "formats":[]}

            # Extract metadata
            meta = ffprobe_metadata(output_path)
            video_stream = next((s for s in meta["streams"] if s["codec_type"]=="video"), None)
            fps = 30
            width = height = ""
            if video_stream:
                width = video_stream.get("width","")
                height = video_stream.get("height","")
                if "r_frame_rate" in video_stream:
                    num, den = map(int, video_stream["r_frame_rate"].split("/"))
                    fps = num/den if den!=0 else 30

            start_ts = frame_to_timestamp(start_frame, fps)
            end_ts = frame_to_timestamp(end_frame, fps)
            duration = round((end_frame - start_frame)/fps,3)

            # Compute checksum
            checksum = sha256_checksum(output_path)

            # Build row dict
            enriched_row = row.to_dict()
            enriched_row.update({
                "title": info.get("title",""),
                "uploader": info.get("uploader",""),
                "fps": fps,
                "start_time": start_ts,
                "end_time": end_ts,
                "duration": duration,
                "checksum": checksum,
                "width": width,
                "height": height
            })

            # Append row immediately to CSV
            append_row_to_csv(enriched_row, OUTPUT_CSV)

        except Exception as e:
            print(f"❌ Row {idx} failed: {e}")

    print(f"\n✅ Finished. Enriched CSV saved to {OUTPUT_CSV}")

# -------------------- Entry Point -------------------

if __name__ == "__main__":
    main()


# Final Downloader - Ignores duplicates and filters for Target Uploader

In [5]:
#!/usr/bin/env python3
"""
YouTube Segment Downloader with incremental CSV enrichment
and uploader filtering. Avoids duplicates on re-runs.
"""

from yt_dlp import YoutubeDL
import os
import subprocess
import pandas as pd
import re
import hashlib
import json
import csv

# -------------------- USER SETTINGS --------------------
INPUT_CSV = "../data/GAVD_data/csv_logs/merged_summary.csv"
OUTPUT_CSV = "../data/GAVD_data/MissionGate/merged_summary_enriched.csv"
TEMP_FOLDER = "../data/GAVD_data/MissionGate/temp_videos"
OUTPUT_VIDEO_FOLDER = "../data/GAVD_data/MissionGate/video_snippets"
TARGET_UPLOADER = "Mission Gait"  # Only download/process videos from this uploader
# ------------------------------------------------------

os.makedirs(TEMP_FOLDER, exist_ok=True)
os.makedirs(OUTPUT_VIDEO_FOLDER, exist_ok=True)

# -------------------- Utilities ----------------------

def safe_name(s):
    return re.sub(r"[^\w\-_. ]", "_", str(s)).strip()

def frame_to_timestamp(frame, fps):
    seconds = frame / fps
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = seconds % 60
    return f"{h:02}:{m:02}:{s:06.3f}"

def sha256_checksum(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

def ffprobe_metadata(file_path):
    cmd = [
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration:stream=codec_type,codec_name,width,height,r_frame_rate",
        "-print_format", "json",
        file_path
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"ffprobe failed on {file_path}")
    return json.loads(result.stdout)

"""
def download_full_video(url):
    ydl_opts = {
        "format": "bv*/b",  # best video or best combined
        "outtmpl": os.path.join(TEMP_FOLDER, "%(title)s.%(ext)s"),
        "noplaylist": True,
        "quiet": False,
        "remote_components": ["ejs:github"],
    }
    with YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=True)

    if info is None:
        raise RuntimeError("yt-dlp returned no info")
    if info.get("vcodec") == "none":
        raise RuntimeError("Audio-only stream — no video available")

    title = safe_name(info["title"])
    ext = info.get("ext")
    input_path = os.path.join(TEMP_FOLDER, f"{title}.{ext}")

    if not os.path.exists(input_path):
        raise FileNotFoundError(f"Downloaded file missing: {input_path}")

    return info, input_path
"""

def get_video_info(url):
    ydl_opts = {
        "quiet": True,
        "noplaylist": True,
        "skip_download": True,
    }
    with YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=False)

    if info is None:
        raise RuntimeError("yt-dlp returned no info")

    return info


def download_video_with_info(info):
    ydl_opts = {
        "format": "bv*/b",
        "outtmpl": os.path.join(TEMP_FOLDER, "%(title)s.%(ext)s"),
        "noplaylist": True,
        "quiet": False,
        "remote_components": ["ejs:github"],
    }
    with YoutubeDL(ydl_opts) as ydl:
        ydl.process_info(info)

    title = safe_name(info["title"])
    ext = info.get("ext")
    input_path = os.path.join(TEMP_FOLDER, f"{title}.{ext}")

    if not os.path.exists(input_path):
        raise FileNotFoundError(f"Downloaded file missing: {input_path}")

    return input_path

    
def cut_and_reencode(input_file, start_ts, end_ts, output_file):
    if not os.path.exists(output_file):
        subprocess.run([
            "ffmpeg", "-y",
            "-i", input_file,
            "-ss", start_ts,
            "-to", end_ts,
            "-c:v", "libx264",
            "-c:a", "aac",
            output_file
        ], check=True)

def append_row_to_csv(row_dict, csv_file):
    file_exists = os.path.exists(csv_file)
    with open(csv_file, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=row_dict.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(row_dict)

# -------------------- Main ---------------------------

def main():
    df = pd.read_csv(INPUT_CSV)

    # Load already processed rows to avoid duplicates
    if os.path.exists(OUTPUT_CSV):
        df_existing = pd.read_csv(OUTPUT_CSV)
        processed_keys = set(
            zip(df_existing["url"], df_existing["start_frame"], df_existing["end_frame"])
        )
    else:
        processed_keys = set()

    # Add enrichment columns if missing
    for col in ["title","uploader","fps","start_time","end_time","duration","checksum","width","height"]:
        if col not in df.columns:
            df[col] = ""

    for idx, row in df.iterrows():
        try:
            

            print(f"\n▶ Processing row {idx}")

            url = row["url"]
            start_frame = int(row["start_frame"])
            end_frame = int(row["end_frame"])

            output_name = safe_name(f"{row['seq']}_{row['cam_view']}_{row['gait_event']}_{row['dataset']}_{row['gait_pat']}.mp4")
            output_path = os.path.join(OUTPUT_VIDEO_FOLDER, output_name)

            # Download video
            """info, input_video = download_full_video(url)
            uploader = info.get("uploader","")
            if uploader != TARGET_UPLOADER:
                print(f"Skipping video '{info.get('title','')}' (Uploader: {uploader})")
                continue"""
            
            # Get metadata ONLY (no download yet)
            info = get_video_info(url)
            uploader = info.get("uploader", "")

            # Skip before download if uploader does not match
            if uploader != TARGET_UPLOADER:
                print(f"Skipping video '{info.get('title','')}' (Uploader: {uploader})")
                continue
            
            #Ensure no duplicated processing of URL frame segments
            key = (row["url"], row["start_frame"], row["end_frame"])
            if key in processed_keys:
                print(f"▶ Row {idx} already processed, skipping")
                continue

            # Download only approved uploader videos
            input_video = download_video_with_info(info)


            # Cut clip
            fps_list = [f.get("fps") for f in info.get("formats", []) if f.get("fps")]
            fps = max(fps_list) if fps_list else 30
            start_ts = frame_to_timestamp(start_frame, fps)
            end_ts = frame_to_timestamp(end_frame, fps)
            cut_and_reencode(input_video, start_ts, end_ts, output_path)

            if os.path.exists(input_video):
                os.remove(input_video)

            # Extract metadata
            meta = ffprobe_metadata(output_path)
            video_stream = next((s for s in meta["streams"] if s["codec_type"]=="video"), None)
            width = height = ""
            if video_stream:
                width = video_stream.get("width","")
                height = video_stream.get("height","")
                if "r_frame_rate" in video_stream:
                    num, den = map(int, video_stream["r_frame_rate"].split("/"))
                    fps = num/den if den!=0 else 30

            start_ts = frame_to_timestamp(start_frame, fps)
            end_ts = frame_to_timestamp(end_frame, fps)
            duration = round((end_frame - start_frame)/fps,3)
            checksum = sha256_checksum(output_path)

            # Build row dict
            enriched_row = row.to_dict()
            enriched_row.update({
                "title": info.get("title",""),
                "uploader": uploader,
                "fps": fps,
                "start_time": start_ts,
                "end_time": end_ts,
                "duration": duration,
                "checksum": checksum,
                "width": width,
                "height": height
            })

            append_row_to_csv(enriched_row, OUTPUT_CSV)
            processed_keys.add(key)

            print(f"✅ Saved clip: {output_path}")

        except Exception as e:
            print(f"❌ Row {idx} failed: {e}")

    print(f"\n✅ Finished. Enriched CSV saved to {OUTPUT_CSV}")

# -------------------- Entry Point -------------------

if __name__ == "__main__":
    main()



▶ Processing row 0
Skipping video 'Parkinsonian Gait Video' (Uploader: Carroll College)

▶ Processing row 1
Skipping video 'Parkinsonian Gait Video' (Uploader: Carroll College)

▶ Processing row 2
[download] ../data/GAVD_data/MissionGate/temp_videos/Brain Injury Gait (AFO) - Case Study 23.mkv has already been downloaded
❌ Row 2 failed: Downloaded file missing: ../data/GAVD_data/MissionGate/temp_videos/Brain Injury Gait _AFO_ - Case Study 23.mkv

▶ Processing row 3
[download] ../data/GAVD_data/MissionGate/temp_videos/Brain Injury Gait (AFO) - Case Study 23.mkv has already been downloaded
❌ Row 3 failed: Downloaded file missing: ../data/GAVD_data/MissionGate/temp_videos/Brain Injury Gait _AFO_ - Case Study 23.mkv

▶ Processing row 4
[download] ../data/GAVD_data/MissionGate/temp_videos/Brain Injury Gait (AFO) - Case Study 23.mkv has already been downloaded
❌ Row 4 failed: Downloaded file missing: ../data/GAVD_data/MissionGate/temp_videos/Brain Injury Gait _AFO_ - Case Study 23.mkv

▶ Pro

Skipping video 'Basics of Walking Gait Evaluation' (Uploader: Richard Blake)

▶ Processing row 16
Skipping video 'Basics of Walking Gait Evaluation' (Uploader: Richard Blake)

▶ Processing row 17
Skipping video 'Basics of Walking Gait Evaluation' (Uploader: Richard Blake)

▶ Processing row 18


Skipping video 'Basics of Walking Gait Evaluation' (Uploader: Richard Blake)

▶ Processing row 19
Skipping video 'Basics of Walking Gait Evaluation' (Uploader: Richard Blake)

▶ Processing row 20
Skipping video 'Basics of Walking Gait Evaluation' (Uploader: Richard Blake)

▶ Processing row 21
Skipping video 'Basics of Walking Gait Evaluation' (Uploader: Richard Blake)

▶ Processing row 22
Skipping video 'Basics of Walking Gait Evaluation' (Uploader: Richard Blake)

▶ Processing row 23
Skipping video 'Basics of Walking Gait Evaluation' (Uploader: Richard Blake)

▶ Processing row 24
Skipping video 'Basics of Walking Gait Evaluation' (Uploader: Richard Blake)

▶ Processing row 25
Skipping video 'Basics of Walking Gait Evaluation' (Uploader: Richard Blake)

▶ Processing row 26
Skipping video 'Basics of Walking Gait Evaluation' (Uploader: Richard Blake)

▶ Processing row 27
Skipping video 'Basics of Walking Gait Evaluation' (Uploader: Richard Blake)

▶ Processing row 28
Skipping video 'Basi

ERROR: [youtube] jzkn287X-84: Video unavailable


❌ Row 43 failed: ERROR: [youtube] jzkn287X-84: Video unavailable

▶ Processing row 44


...

❌ Row 1878 failed: ERROR: [youtube] FFki8FtaByw: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies

▶ Processing row 1879


ERROR: [youtube] Ha9LKXZfWBQ: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


❌ Row 1879 failed: ERROR: [youtube] Ha9LKXZfWBQ: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies

▶ Processing row 1880


ERROR: [youtube] Ha9LKXZfWBQ: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


❌ Row 1880 failed: ERROR: [youtube] Ha9LKXZfWBQ: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies

✅ Finished. Enriched CSV saved to ../data/GAVD_data/MissionGate/merged_summary_enriched.csv


## Running scripts for video and csv extraction only - manual input

In [ ]:
#!/usr/bin/env python3
"""
YouTube QuickTime-Compatible Segment Downloader by Frame Number
Requirements:
- Python 3
- yt-dlp (`pip install yt-dlp`)
- ffmpeg installed and in PATH
- Optional: Deno installed for JS challenges
"""

from yt_dlp import YoutubeDL
import os
import subprocess
import sys

# -------------------- USER SETTINGS --------------------
URLS_FILE = "../data/videourls.txt"         # Text file with YouTube URLs
TARGET_UPLOADER = "Mission Gait"            # Only download videos from this uploader
START_FRAME = 1000                           # Start frame
END_FRAME = 1300                             # End frame
OUTPUT_TEMPLATE = "%(title)s_%(section_start)s-%(section_end)s.mp4"
TEMP_FOLDER = "temp_videos"                  # Temporary folder for full downloads
# -------------------------------------------------------

def frame_to_timestamp(frame, fps):
    """Convert frame number to HH:MM:SS.sss timestamp."""
    seconds = frame / fps
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = seconds % 60
    return f"{h:02}:{m:02}:{s:06.3f}"

def ensure_folder(folder):
    if not os.path.exists(folder):
        os.makedirs(folder)

def download_full_video(url, temp_folder):
    """Download best MP4 video + audio, merged into MP4. Skip if unavailable."""
    ydl_opts = {
        "format": "bestvideo[ext=mp4]+bestaudio[ext=m4a]/bestvideo+bestaudio/best",
        "merge_output_format": "mp4",
        "outtmpl": os.path.join(temp_folder, "%(title)s.%(ext)s"),
        "noplaylist": True,
        "ignoreerrors": True,
        "no_warnings": True,
        "quiet": False,
        "remote_components": "ejs:github",  # solves JS challenges
    }
    with YoutubeDL(ydl_opts) as ydl:
        try:
            info = ydl.extract_info(url, download=True)
            if info is None:
                print(f"Skipping {url} — no suitable video format available")
                return None
            return info
        except Exception as e:
            print(f"Error downloading {url}: {e}")
            return None

def cut_video_segment(input_file, start_ts, end_ts, output_file):
    """Cut a segment and re-encode to QuickTime-compatible MP4 (H.264 + AAC)."""
    if not os.path.exists(input_file):
        raise FileNotFoundError(f"Input file not found: {input_file}")
    
    cmd = [
        "ffmpeg",
        "-y",
        "-i", input_file,
        "-ss", start_ts,
        "-to", end_ts,
        "-c:v", "libx264",
        "-c:a", "aac",
        "-strict", "experimental",
        output_file
    ]
    subprocess.run(cmd, check=True)

def process_video(url, start_frame, end_frame, target_uploader, output_template, temp_folder):
    info = download_full_video(url, temp_folder)
    if info is None:
        return  # Skip this video

    uploader = info.get("uploader")
    if uploader != target_uploader:
        print(f"Skipping '{info.get('title', 'Unknown')}' (Uploader: {uploader})")
        return

    # Determine FPS safely
    fps_list = [f.get("fps") for f in info.get("formats", []) if f.get("fps")]
    fps = max(fps_list) if fps_list else 30
    print(f"Processing '{info.get('title', 'Unknown')}' (FPS: {fps}, Uploader: {uploader})")

    # Convert frames to timestamps
    start_ts = frame_to_timestamp(start_frame, fps)
    end_ts = frame_to_timestamp(end_frame, fps)

    # Determine downloaded file path (MP4)
    input_file = os.path.join(temp_folder, f"{info['title']}.mp4")
    if not os.path.exists(input_file):
        # Try original extension if MP4 not available
        ext = info.get("ext") or "mp4"
        input_file = os.path.join(temp_folder, f"{info['title']}.{ext}")
        if not os.path.exists(input_file):
            print(f"Skipping '{info['title']}' — video file not found")
            return

    # Build output filename
    output_file = output_template.replace("%(title)s", info['title'])\
                                 .replace("%(section_start)s", start_ts)\
                                 .replace("%(section_end)s", end_ts)

    # Cut segment and re-encode to QuickTime-compatible MP4
    cut_video_segment(input_file, start_ts, end_ts, output_file)
    print(f"Saved segment: {output_file}")

    # Delete temp full video
    if os.path.exists(input_file):
        os.remove(input_file)

def main():
    ensure_folder(TEMP_FOLDER)

    # Load URLs
    try:
        with open(URLS_FILE, "r") as f:
            urls = [line.strip() for line in f if line.strip()]
    except FileNotFoundError:
        print(f"Error: File '{URLS_FILE}' not found.")
        sys.exit(1)

    # Process each video
    for url in urls:
        try:
            process_video(url, START_FRAME, END_FRAME, TARGET_UPLOADER, OUTPUT_TEMPLATE, TEMP_FOLDER)
        except Exception as e:
            print(f"Error processing {url}: {e}")

if __name__ == "__main__":
    main()


In [ ]:
#!/usr/bin/env python3
"""
YouTube QuickTime-Compatible Segment Downloader by Frame Number
With CSV logging of video info.

Requirements:
- Python 3
- yt-dlp (`pip install yt-dlp`)
- ffmpeg installed and in PATH
- Optional: Deno installed for JS challenges
"""

from yt_dlp import YoutubeDL
import os
import subprocess
import sys
import csv

# -------------------- USER SETTINGS --------------------
URLS_FILE = "../data/videourls.txt"         # Text file with YouTube URLs
TARGET_UPLOADER = "Mission Gait"            # Only download videos from this uploader
START_FRAME = 1000                           # Start frame
END_FRAME = 1300                             # End frame
OUTPUT_TEMPLATE = "%(title)s_%(section_start)s-%(section_end)s.mp4"
TEMP_FOLDER = "temp_videos"                  # Temporary folder for full downloads
CSV_LOG_FILE = "video_log.csv"               # CSV file to save segment info
# -------------------------------------------------------

def frame_to_timestamp(frame, fps):
    """Convert frame number to HH:MM:SS.sss timestamp."""
    seconds = frame / fps
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = seconds % 60
    return f"{h:02}:{m:02}:{s:06.3f}"

def ensure_folder(folder):
    if not os.path.exists(folder):
        os.makedirs(folder)

def download_full_video(url, temp_folder):
    """Download best MP4 video + audio, merged into MP4. Skip if unavailable."""
    ydl_opts = {
        "format": "bestvideo[ext=mp4]+bestaudio[ext=m4a]/bestvideo+bestaudio/best",
        "merge_output_format": "mp4",
        "outtmpl": os.path.join(temp_folder, "%(title)s.%(ext)s"),
        "noplaylist": True,
        "ignoreerrors": True,
        "no_warnings": True,
        "quiet": False,
        "remote_components": "ejs:github",  # solves JS challenges
    }
    with YoutubeDL(ydl_opts) as ydl:
        try:
            info = ydl.extract_info(url, download=True)
            if info is None:
                print(f"Skipping {url} — no suitable video format available")
                return None
            return info
        except Exception as e:
            print(f"Error downloading {url}: {e}")
            return None

def cut_video_segment(input_file, start_ts, end_ts, output_file):
    """Cut a segment and re-encode to QuickTime-compatible MP4 (H.264 + AAC)."""
    if not os.path.exists(input_file):
        raise FileNotFoundError(f"Input file not found: {input_file}")
    
    cmd = [
        "ffmpeg",
        "-y",
        "-i", input_file,
        "-ss", start_ts,
        "-to", end_ts,
        "-c:v", "libx264",
        "-c:a", "aac",
        "-strict", "experimental",
        output_file
    ]
    subprocess.run(cmd, check=True)

def log_to_csv(row, csv_file):
    """Append a row to CSV, create file if it doesn't exist."""
    file_exists = os.path.exists(csv_file)
    with open(csv_file, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=row.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)

def process_video(url, start_frame, end_frame, target_uploader, output_template, temp_folder, csv_file):
    info = download_full_video(url, temp_folder)
    if info is None:
        return  # Skip this video

    uploader = info.get("uploader")
    if uploader != target_uploader:
        print(f"Skipping '{info.get('title', 'Unknown')}' (Uploader: {uploader})")
        return

    # Determine FPS safely
    fps_list = [f.get("fps") for f in info.get("formats", []) if f.get("fps")]
    fps = max(fps_list) if fps_list else 30
    print(f"Processing '{info.get('title', 'Unknown')}' (FPS: {fps}, Uploader: {uploader})")

    # Convert frames to timestamps
    start_ts = frame_to_timestamp(start_frame, fps)
    end_ts = frame_to_timestamp(end_frame, fps)
    duration = round((end_frame - start_frame) / fps, 3)

    # Determine downloaded file path (MP4)
    input_file = os.path.join(temp_folder, f"{info['title']}.mp4")
    if not os.path.exists(input_file):
        ext = info.get("ext") or "mp4"
        input_file = os.path.join(temp_folder, f"{info['title']}.{ext}")
        if not os.path.exists(input_file):
            print(f"Skipping '{info['title']}' — video file not found")
            return

    # Build output filename
    output_file = output_template.replace("%(title)s", info['title'])\
                                 .replace("%(section_start)s", start_ts)\
                                 .replace("%(section_end)s", end_ts)

    # Cut segment and re-encode to QuickTime-compatible MP4
    cut_video_segment(input_file, start_ts, end_ts, output_file)
    print(f"Saved segment: {output_file}")

    # Delete temp full video
    if os.path.exists(input_file):
        os.remove(input_file)

    # Log info to CSV
    row = {
        "title": info['title'],
        "url": url,
        "uploader": uploader,
        "fps": fps,
        "start_frame": start_frame,
        "end_frame": end_frame,
        "start_time": start_ts,
        "end_time": end_ts,
        "duration": duration
    }
    log_to_csv(row, csv_file)

def main():
    ensure_folder(TEMP_FOLDER)

    # Load URLs
    try:
        with open(URLS_FILE, "r") as f:
            urls = [line.strip() for line in f if line.strip()]
    except FileNotFoundError:
        print(f"Error: File '{URLS_FILE}' not found.")
        sys.exit(1)

    # Process each video
    for url in urls:
        try:
            process_video(url, START_FRAME, END_FRAME, TARGET_UPLOADER, OUTPUT_TEMPLATE, TEMP_FOLDER, CSV_LOG_FILE)
        except Exception as e:
            print(f"Error processing {url}: {e}")

if __name__ == "__main__":
    main()
